<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

### 📊 Resumo de Metadados: Variáveis ARCO-ERA5

| Grandeza | Nome Longo | Nome Curto | Unidade | Shape (Time, Lat, Lon) |
| :--- | :--- | :--- | :--- | :--- |
| **Temperature (2m)** | 2 metre temperature | `t2m` | K | [1323648, 721, 1440] |
| **Dewpoint Temp. (2m)** | 2 metre dewpoint temperature | `d2m` | K | [1323648, 721, 1440] |
| **U Wind Component (10m)** | 10 metre U wind component | `u10` | m s**-1 | [1323648, 721, 1440] |
| **V Wind Component (10m)** | 10 metre V wind component | `v10` | m s**-1 | [1323648, 721, 1440] |
| **Total Precipitation** | Total precipitation | `tp` | m | [1323648, 721, 1440] |
| **Solar Radiation** | Surface solar radiation downwards | `ssrd` | J m**-2 | [1323648, 721, 1440] |

**Coordenadas Globais:**
*   **Latitude:** -90.00° a 90.00° (Ordem Decrescente)
*   **Longitude:** 0.00° a 359.75°

In [ ]:
!pip install geodatasets geopandas xarray dask rioxarray zarr -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil
import glob
import gc
import logging
import numpy as np
import xarray as xr
import dask
import geopandas as gpd
import rioxarray
from tqdm.auto import tqdm

# ==============================================================================
# 1. PAINEL DE CONTROLE (ABA 1: TEMPERATURAS E HURS)
# ==============================================================================
dask.config.set(scheduler='single-threaded')
# Evitar que o Dask reclame de fragmentação de blocos
dask.config.set({"array.slicing.split_large_chunks": True})
logging.basicConfig(level=logging.WARNING, format='%(message)s')

ZARR_URL = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
ANOS = range(1940, 2026)

VARIAVEIS_ALVO = ['tasmax', 'tasmin', 'hurs']

DIRETORIO_DRIVE = '/content/drive/Shareddrives/GAS-Henrique/ERA5_Recortado'
DIRETORIO_LOCAL = '/content/ERA5_Export_Local'
TEMP_DIR = '/content/temp_era5'
CAMINHO_SHAPEFILE = '/content/drive/Shareddrives/GAS-Henrique/shapefiles.shp'

MAPA_DEPENDENCIAS = {
    'tasmax': ['2m_temperature'],
    'tasmin': ['2m_temperature'],
    'hurs': ['2m_temperature', '2m_dewpoint_temperature']
}

vars_to_load_set = set()
for var in VARIAVEIS_ALVO:
    vars_to_load_set.update(MAPA_DEPENDENCIAS[var])
VARIABLES_TO_LOAD = list(vars_to_load_set)

# ==============================================================================
# 2. PREPARAÇÃO DE PASTAS E GEOGRAFIA
# ==============================================================================
print(f"Aba configurada para: {VARIAVEIS_ALVO}")
print("A carregar o shapefile...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)

for var in VARIAVEIS_ALVO:
    os.makedirs(os.path.join(DIRETORIO_DRIVE, var), exist_ok=True)
    os.makedirs(os.path.join(DIRETORIO_LOCAL, var), exist_ok=True)

def limpar_temp():
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=True)
    os.makedirs(TEMP_DIR, exist_ok=True)
    for dask_trash in glob.glob('/tmp/dask-worker-space*'):
        try: shutil.rmtree(dask_trash, ignore_errors=True)
        except: pass

def formatar_dataset_saida(da, var_name, units, gdf_mask):
    ds_out = xr.Dataset({var_name: da.astype('float32')})
    ds_out[var_name].attrs['units'] = units
    ds_out.coords['longitude'] = (ds_out.coords['longitude'] + 180) % 360 - 180
    ds_out = ds_out.sortby(ds_out.longitude)
    ds_out = ds_out.sortby(ds_out.latitude)
    ds_out = ds_out.rename({'latitude': 'lat', 'longitude': 'lon'})
    ds_out.rio.write_crs("epsg:4326", inplace=True)
    ds_out.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    ds_out = ds_out.rio.clip(gdf_mask.geometry, gdf_mask.crs, drop=True)
    return ds_out

# ==============================================================================
# 3. LOOP PRINCIPAL (PROCESSAMENTO MENSAL PARA POUPAR RAM)
# ==============================================================================
try:
    print("\nA mapear ARCO-ERA5 via GCS (Lazy Load)...")
    ds_global = xr.open_zarr(
        ZARR_URL,
        consolidated=True,
        storage_options={'token': 'anon'},
        chunks={'time': 24} # Lendo dia a dia
    )[VARIABLES_TO_LOAD]

    pbar_anos = tqdm(ANOS, desc="Progresso Global", unit="ano")

    for ano in pbar_anos:
        # 1. VERIFICAÇÃO DE CHECKPOINT ANUAL
        ano_completo = True
        for var_alvo in VARIAVEIS_ALVO:
            arq_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            if not (os.path.exists(arq_drive) and os.path.getsize(arq_drive) > 10240):
                ano_completo = False
                break

        if ano_completo:
            pbar_anos.set_postfix_str(f"Ano {ano} já no Drive. Pulando...")
            continue

        # 2. O LOOP MENSAL (A SALVAÇÃO DA MEMÓRIA RAM)
        pbar_meses = tqdm(range(1, 13), desc=f"Processando {ano} (Meses)", leave=False, unit="mês")

        for mes in pbar_meses:
            mes_str = f"{mes:02d}"

            # Recorta apenas 1 mês (Ex: 1940-01) - Uso de RAM fica minúsculo
            ds_mes = ds_global.sel(time=f'{ano}-{mes_str}')
            if len(ds_mes.time) == 0:
                continue

            # Prepara as fórmulas matemáticas só para este mês
            daily_max = ds_mes['2m_temperature'].resample(time='1D').max(skipna=False)
            daily_min = ds_mes['2m_temperature'].resample(time='1D').min(skipna=False)

            vars_mean = [v for v in ['2m_dewpoint_temperature'] if v in ds_mes]
            daily_mean = ds_mes[vars_mean].resample(time='1D').mean(skipna=False) if vars_mean else None

            # Calcula e salva fragmentos mensais para cada variável
            for var_alvo in VARIAVEIS_ALVO:
                arq_mes = os.path.join(DIRETORIO_LOCAL, f"temp_{var_alvo}_{ano}_{mes_str}.nc")

                # Se o arquivo final do ano já existir no Drive, ignora o mês
                arq_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
                if os.path.exists(arq_drive) and os.path.getsize(arq_drive) > 10240:
                    continue

                pbar_meses.set_postfix_str(f"A calcular {var_alvo} ({mes_str}/{ano})...")

                if var_alvo == 'tasmax':
                    da = daily_max - 273.15
                    ds_out = formatar_dataset_saida(da, var_alvo, 'degC', gdf_continentes)
                elif var_alvo == 'tasmin':
                    da = daily_min - 273.15
                    ds_out = formatar_dataset_saida(da, var_alvo, 'degC', gdf_continentes)
                elif var_alvo == 'hurs':
                    tasmax_k = daily_max
                    tasmin_k = daily_min
                    Tdew_c = daily_mean['2m_dewpoint_temperature'] - 273.15
                    Ea = 0.6108 * np.exp((17.27 * Tdew_c) / (Tdew_c + 237.3))
                    Es_tmax = 0.6108 * np.exp((17.27 * (tasmax_k - 273.15)) / ((tasmax_k - 273.15) + 237.3))
                    Es_tmin = 0.6108 * np.exp((17.27 * (tasmin_k - 273.15)) / ((tasmin_k - 273.15) + 237.3))
                    Es = (Es_tmax + Es_tmin) / 2.0
                    da = 100 * (Ea / Es)
                    da = da.clip(min=0, max=100)
                    ds_out = formatar_dataset_saida(da, var_alvo, '%', gdf_continentes)

                # Salva o pedaço do mês no disco local (Rápido e sem compressão para não pesar)
                ds_out.to_netcdf(arq_mes, engine='h5netcdf')

                del da, ds_out
                gc.collect()

            # Limpa o mês da RAM
            del ds_mes, daily_max, daily_min, daily_mean
            gc.collect()

        # 3. COSTURAR OS 12 MESES E ENVIAR PARA O DRIVE
        for var_alvo in VARIAVEIS_ALVO:
            arq_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            arq_local_anual = os.path.join(DIRETORIO_LOCAL, f"ERA5_{var_alvo}_{ano}.nc4")

            if os.path.exists(arq_drive) and os.path.getsize(arq_drive) > 10240:
                continue

            arquivos_mensais = sorted(glob.glob(os.path.join(DIRETORIO_LOCAL, f"temp_{var_alvo}_{ano}_*.nc")))
            if not arquivos_mensais:
                continue

            pbar_anos.set_postfix_str(f"A compilar e comprimir {var_alvo} ({ano})...")

            # Abre os 12 pedacinhos juntos
            ds_anual = xr.open_mfdataset(arquivos_mensais, combine='by_coords', engine='h5netcdf')

            # Configura a compressão final
            encoding = {var_alvo: {'zlib': True, 'complevel': 5, '_FillValue': np.nan}}

            # Grava o arquivo anual consolidado no local
            ds_anual.to_netcdf(arq_local_anual, engine='h5netcdf', encoding=encoding)

            ds_anual.close()
            del ds_anual
            gc.collect()

            # Transfere para o Drive
            pbar_anos.set_postfix_str(f"Enviando {var_alvo} para o Drive...")
            shutil.move(arq_local_anual, arq_drive)

        # 4. LIMPEZA TOTAL DO ANO NO DISCO LOCAL
        for f in glob.glob(os.path.join(DIRETORIO_LOCAL, f"temp_*_{ano}_*.nc")):
            os.remove(f)
        gc.collect()

except KeyboardInterrupt:
    print("\n⛔ Interrupção manual. O progresso consolidado está a salvo no Drive.")
except Exception as e:
    print(f"\n❌ Erro crítico GCS: {e}")
finally:
    if 'ds_global' in locals(): del ds_global
    limpar_temp()
    shutil.rmtree(DIRETORIO_LOCAL, ignore_errors=True)
    gc.collect()

Aba configurada para: ['tasmax', 'tasmin', 'hurs']
A carregar o shapefile...

A mapear ARCO-ERA5 via GCS (Lazy Load)...


Progresso Global:   0%|          | 0/86 [00:00<?, ?ano/s]

Processando 1989 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 1990 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 1991 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

...

Processando 2021 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 2022 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 2023 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 2024 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

Processando 2025 (Meses):   0%|          | 0/12 [00:00<?, ?mês/s]

In [ ]:
import os
import shutil
import glob
import gc
import logging
import numpy as np
import xarray as xr
import dask
import geopandas as gpd
import rioxarray
from tqdm.auto import tqdm

# ==============================================================================
# 1. PAINEL DE CONTROLO (ABA 2: VENTO)
# ==============================================================================
dask.config.set(scheduler='single-threaded')
logging.basicConfig(level=logging.WARNING, format='%(message)s')

ZARR_URL = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
ANOS = range(1940, 2026)

VARIAVEIS_ALVO = ['sfcWind']

DIRETORIO_DRIVE = '/content/drive/Shareddrives/GAS-Henrique/ERA5_Recortado'
DIRETORIO_LOCAL = '/content/ERA5_Export_Local'
TEMP_DIR = '/content/temp_era5'
CAMINHO_SHAPEFILE = '/content/drive/Shareddrives/GAS-Henrique/shapefiles.shp'

MAPA_DEPENDENCIAS = {
    'sfcWind': ['10m_u_component_of_wind', '10m_v_component_of_wind']
}

vars_to_load_set = set()
for var in VARIAVEIS_ALVO:
    vars_to_load_set.update(MAPA_DEPENDENCIAS[var])
VARIABLES_TO_LOAD = list(vars_to_load_set)

# ==============================================================================
# 2. PREPARAÇÃO DE PASTAS E GEOGRAFIA
# ==============================================================================
print(f"Aba 2 configurada para: {VARIAVEIS_ALVO}")
print("A carregar o shapefile...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)

for var in VARIAVEIS_ALVO:
    os.makedirs(os.path.join(DIRETORIO_DRIVE, var), exist_ok=True)
    os.makedirs(os.path.join(DIRETORIO_LOCAL, var), exist_ok=True)

def limpar_temp():
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=True)
    os.makedirs(TEMP_DIR, exist_ok=True)
    for dask_trash in glob.glob('/tmp/dask-worker-space*'):
        try: shutil.rmtree(dask_trash, ignore_errors=True)
        except: pass

def wind10to2(wind10):
    fator = np.log10(2.0 / 0.033) / np.log10(10.0 / 0.033)
    return wind10 * fator

def formatar_dataset_saida(da, var_name, units, gdf_mask):
    ds_out = xr.Dataset({var_name: da.astype('float32')})
    ds_out[var_name].attrs['units'] = units
    ds_out.coords['longitude'] = (ds_out.coords['longitude'] + 180) % 360 - 180
    ds_out = ds_out.sortby(ds_out.longitude)
    ds_out = ds_out.sortby(ds_out.latitude)
    ds_out = ds_out.rename({'latitude': 'lat', 'longitude': 'lon'})
    ds_out.rio.write_crs("epsg:4326", inplace=True)
    ds_out.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    ds_out = ds_out.rio.clip(gdf_mask.geometry, gdf_mask.crs, drop=True)
    ds_out[var_name].encoding = {'zlib': True, 'complevel': 5, '_FillValue': np.nan}
    return ds_out

# ==============================================================================
# 3. LOOP PRINCIPAL
# ==============================================================================
try:
    print("\nA mapear ARCO-ERA5 via GCS (Lazy Load)...")
    ds_global = xr.open_zarr(
        ZARR_URL,
        consolidated=True,
        storage_options={'token': 'anon'},
        chunks={'time': 24}
    )[VARIABLES_TO_LOAD]

    pbar_anos = tqdm(ANOS, desc="Progresso Global", unit="ano")

    for ano in pbar_anos:
        ano_completo = True
        for var_alvo in VARIAVEIS_ALVO:
            arq_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            if not (os.path.exists(arq_drive) and os.path.getsize(arq_drive) > 10240):
                ano_completo = False
                break

        if ano_completo:
            pbar_anos.set_postfix_str(f"Ano {ano} já no Drive. Pulando...")
            continue

        pbar_anos.set_postfix_str(f"A fatiar e agregar {ano}...")
        ds_ano = ds_global.sel(time=slice(f'{ano}-01-01', f'{ano}-12-31'))

        if len(ds_ano.time) == 0:
            continue

        # Faz apenas a média para o vento
        vars_mean = ['10m_u_component_of_wind', '10m_v_component_of_wind']
        daily_mean = ds_ano[vars_mean].resample(time='1D').mean(skipna=False)

        pbar_vars = tqdm(VARIAVEIS_ALVO, desc=f"Variáveis {ano}", leave=False, unit="var")

        for var_alvo in pbar_vars:
            arquivo_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            arquivo_local = os.path.join(DIRETORIO_LOCAL, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")

            if os.path.exists(arquivo_drive) and os.path.getsize(arquivo_drive) > 10240:
                pbar_vars.set_postfix_str(f"Pulado")
                continue

            pbar_vars.set_postfix_str(f"A calcular...")
            limpar_temp()
            ds_out = None

            try:
                u = daily_mean['10m_u_component_of_wind']
                v = daily_mean['10m_v_component_of_wind']
                wind10 = np.sqrt(u**2 + v**2)
                da = wind10to2(wind10)
                da = da.clip(min=0)
                ds_out = formatar_dataset_saida(da, var_alvo, 'm s-1', gdf_continentes)

                pbar_vars.set_postfix_str(f"A comprimir e gravar localmente...")
                ds_out.to_netcdf(arquivo_local, engine='h5netcdf', format='NETCDF4')

                pbar_vars.set_postfix_str(f"A transferir para o Drive...")
                shutil.move(arquivo_local, arquivo_drive)

            except Exception as e:
                tqdm.write(f" ❌ Falha no processamento de {var_alvo} ({ano}): {e}")
                if os.path.exists(arquivo_local): os.remove(arquivo_local)
                if os.path.exists(arquivo_drive): os.remove(arquivo_drive)
            finally:
                if 'da' in locals(): del da
                if 'ds_out' in locals(): del ds_out
                gc.collect()

        del ds_ano, daily_mean
        gc.collect()

except KeyboardInterrupt:
    print("\n⛔ Interrupção manual. Checkpoints preservados no Drive.")
except Exception as e:
    print(f"\n❌ Erro crítico GCS: {e}")
finally:
    if 'ds_global' in locals(): del ds_global
    limpar_temp()
    shutil.rmtree(DIRETORIO_LOCAL, ignore_errors=True)
    gc.collect()

In [ ]:
import os
import shutil
import glob
import gc
import logging
import numpy as np
import xarray as xr
import dask
import geopandas as gpd
import rioxarray
from tqdm.auto import tqdm

# ==============================================================================
# 1. PAINEL DE CONTROLO (ABA 3: CHUVA E RADIAÇÃO)
# ==============================================================================
dask.config.set(scheduler='single-threaded')
logging.basicConfig(level=logging.WARNING, format='%(message)s')

ZARR_URL = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'
ANOS = range(1940, 2026)

VARIAVEIS_ALVO = ['pr', 'rsds']

DIRETORIO_DRIVE = '/content/drive/Shareddrives/GAS-Henrique/ERA5_Recortado'
DIRETORIO_LOCAL = '/content/ERA5_Export_Local'
TEMP_DIR = '/content/temp_era5'
CAMINHO_SHAPEFILE = '/content/drive/Shareddrives/GAS-Henrique/shapefiles.shp'

MAPA_DEPENDENCIAS = {
    'pr': ['total_precipitation'],
    'rsds': ['surface_solar_radiation_downwards']
}

vars_to_load_set = set()
for var in VARIAVEIS_ALVO:
    vars_to_load_set.update(MAPA_DEPENDENCIAS[var])
VARIABLES_TO_LOAD = list(vars_to_load_set)

# ==============================================================================
# 2. PREPARAÇÃO DE PASTAS E GEOGRAFIA
# ==============================================================================
print(f"Aba 3 configurada para: {VARIAVEIS_ALVO}")
print("A carregar o shapefile...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)

for var in VARIAVEIS_ALVO:
    os.makedirs(os.path.join(DIRETORIO_DRIVE, var), exist_ok=True)
    os.makedirs(os.path.join(DIRETORIO_LOCAL, var), exist_ok=True)

def limpar_temp():
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR, ignore_errors=True)
    os.makedirs(TEMP_DIR, exist_ok=True)
    for dask_trash in glob.glob('/tmp/dask-worker-space*'):
        try: shutil.rmtree(dask_trash, ignore_errors=True)
        except: pass

def formatar_dataset_saida(da, var_name, units, gdf_mask):
    ds_out = xr.Dataset({var_name: da.astype('float32')})
    ds_out[var_name].attrs['units'] = units
    ds_out.coords['longitude'] = (ds_out.coords['longitude'] + 180) % 360 - 180
    ds_out = ds_out.sortby(ds_out.longitude)
    ds_out = ds_out.sortby(ds_out.latitude)
    ds_out = ds_out.rename({'latitude': 'lat', 'longitude': 'lon'})
    ds_out.rio.write_crs("epsg:4326", inplace=True)
    ds_out.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    ds_out = ds_out.rio.clip(gdf_mask.geometry, gdf_mask.crs, drop=True)
    ds_out[var_name].encoding = {'zlib': True, 'complevel': 5, '_FillValue': np.nan}
    return ds_out

# ==============================================================================
# 3. LOOP PRINCIPAL
# ==============================================================================
try:
    print("\nA mapear ARCO-ERA5 via GCS (Lazy Load)...")
    ds_global = xr.open_zarr(
        ZARR_URL,
        consolidated=True,
        storage_options={'token': 'anon'},
        chunks={'time': 24}
    )[VARIABLES_TO_LOAD]

    pbar_anos = tqdm(ANOS, desc="Progresso Global", unit="ano")

    for ano in pbar_anos:
        ano_completo = True
        for var_alvo in VARIAVEIS_ALVO:
            arq_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            if not (os.path.exists(arq_drive) and os.path.getsize(arq_drive) > 10240):
                ano_completo = False
                break

        if ano_completo:
            pbar_anos.set_postfix_str(f"Ano {ano} já no Drive. Pulando...")
            continue

        pbar_anos.set_postfix_str(f"A fatiar e agregar {ano}...")
        ds_ano = ds_global.sel(time=slice(f'{ano}-01-01', f'{ano}-12-31'))

        if len(ds_ano.time) == 0:
            continue

        # Faz apenas a soma diária para precipitação e radiação
        vars_sum = ['total_precipitation', 'surface_solar_radiation_downwards']
        daily_sum = ds_ano[vars_sum].resample(time='1D').sum(skipna=False)

        pbar_vars = tqdm(VARIAVEIS_ALVO, desc=f"Variáveis {ano}", leave=False, unit="var")

        for var_alvo in pbar_vars:
            arquivo_drive = os.path.join(DIRETORIO_DRIVE, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")
            arquivo_local = os.path.join(DIRETORIO_LOCAL, var_alvo, f"ERA5_{var_alvo}_{ano}.nc4")

            if os.path.exists(arquivo_drive) and os.path.getsize(arquivo_drive) > 10240:
                pbar_vars.set_postfix_str(f"Pulado")
                continue

            pbar_vars.set_postfix_str(f"A calcular...")
            limpar_temp()
            ds_out = None

            try:
                if var_alvo == 'rsds':
                    da = daily_sum['surface_solar_radiation_downwards'] / 1e6
                    da = da.clip(min=0)
                    ds_out = formatar_dataset_saida(da, var_alvo, 'MJ m-2 day-1', gdf_continentes)
                elif var_alvo == 'pr':
                    da = daily_sum['total_precipitation'] * 1000.0
                    da = da.clip(min=0)
                    ds_out = formatar_dataset_saida(da, var_alvo, 'mm/day', gdf_continentes)

                pbar_vars.set_postfix_str(f"A comprimir e gravar localmente...")
                ds_out.to_netcdf(arquivo_local, engine='h5netcdf', format='NETCDF4')

                pbar_vars.set_postfix_str(f"A transferir para o Drive...")
                shutil.move(arquivo_local, arquivo_drive)

            except Exception as e:
                tqdm.write(f" ❌ Falha no processamento de {var_alvo} ({ano}): {e}")
                if os.path.exists(arquivo_local): os.remove(arquivo_local)
                if os.path.exists(arquivo_drive): os.remove(arquivo_drive)
            finally:
                if 'da' in locals(): del da
                if 'ds_out' in locals(): del ds_out
                gc.collect()

        del ds_ano, daily_sum
        gc.collect()

except KeyboardInterrupt:
    print("\n⛔ Interrupção manual. Checkpoints preservados no Drive.")
except Exception as e:
    print(f"\n❌ Erro crítico GCS: {e}")
finally:
    if 'ds_global' in locals(): del ds_global
    limpar_temp()
    shutil.rmtree(DIRETORIO_LOCAL, ignore_errors=True)
    gc.collect()